# Invert for source depth / range

In [ ]:
import os
import pickle
from itertools import combinations

import cmps
import matplotlib.pyplot as plt
import numpy as np
import pygenray as pr
import xarray as xr
from dotenv import find_dotenv, load_dotenv
from geopy.distance import geodesic

from nescafe23_fluctuations import tools
from nescafe23_fluctuations.source_localization import source_localization_iteration, source_localization_iteration_waterdepth
from nescafe23_fluctuations import source_localization as sl

_ = load_dotenv(find_dotenv())

%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'retina'

## Try inversion with water depth

In [ ]:
## Open Observed Arrival Times
Tx_idx = 0

with open(
    f"{os.environ['NESCAFE23_DATADIR']}dataproducts/peak_locations.pkl", "rb"
) as f:
    locations = pickle.load(f)

times = locations["times"]
angles = locations["angles"]
snrs = locations["snrs"]


ray_ids = list(times.keys())[7:-6]
difference_keys = list(combinations(ray_ids, 2))

Delta_Ts_obs = np.ones(len(difference_keys)) * np.nan

for k, diff_key in enumerate(difference_keys):
    Delta_Ts_obs[k] = times[diff_key[0]][Tx_idx] - times[diff_key[1]][Tx_idx]

In [ ]:
## Create model environment object
full_field = tools.compute_filtered_fctd_grid(z_smooth=300, r_smooth=5000)

front_location = 35e3

envi_field = full_field.assign_coords({
    "range": (full_field.range.values + front_location)
})
c = envi_field["c"]

bathy = xr.DataArray(
    np.linspace(4908, 4908, 100),
    dims=["range"],
    coords={"range": np.linspace(0, 200e3, 100)},
)

envi = pr.OceanEnvironment2D(sound_speed=c, bathymetry=bathy)

In [ ]:
## Initial Guess and Priors

source_depth0 = 1000  # [m]
reciever_range0 = 153e3  # [m]
reciever_depth0 = 1000  # [m]
water_depth0 = 5000 # [m]

S = np.diag([500, 4000, 500, 200]) ** 2

# 500 [m] for source depth
# 1 [km] for range
# 500 [m] for reciever range
# 500 [m] for water depth

R = 2 * np.diag([(20e-3)] * len(difference_keys)) ** 2
# observation error of 20 [ms]

## Water Depth Inversion

In [ ]:
x0 = np.array([source_depth0, reciever_range0, reciever_depth0, water_depth0])

Delta_Ts_sim = []
E = []
y = []
dx = []
P = []
rays = []
eigs = []
envis = []
x = [x0]

for k in range(10):
    print(
        f"Iteration {k}: Source Depth = {x[k][0]}, Range = {x[k][1]}, Reciever Depth = {x[k][2]}, Water Depth = {x[k][3]}"
    )
    
    Delta_Ts_simi, Ei, yi, dxi, Pi, raysi, eigsi, envii= source_localization_iteration_waterdepth(
        x[k], S, R, envi, ray_ids, Delta_Ts_obs
    )

    Delta_Ts_sim.append(Delta_Ts_simi)
    E.append(Ei)
    y.append(yi)
    dx.append(dxi)
    P.append(Pi)
    rays.append(raysi)
    eigs.append(eigsi)
    envis.append(envii)
    
    # compute update for next iteration
    x.append(x[k] + dx[k])

In [ ]:
for k in range(5):
    print(float(envis[-k].bathymetry.mean()))

In [ ]:
S

In [ ]:
fig, ax = plt.subplots(1,1)

_ = sl.plot_marginal_gaussian(x[0], S, (0, 3), plot_type='contour', cmap='mako_r', ax=ax)
_ = sl.plot_marginal_gaussian(x[-1], P[-1], (0, 3), plot_type='contour', cmap='rocket_r', ax=ax)

In [ ]:
plt.plot(y[-1])
plt.plot(y[-2])
plt.plot(y[-3])
plt.plot(y[-4])

In [ ]:
source_depths = []
ranges = []
r_depths = []
water_depths = []

for k in range(10):
    source_depths.append(x[k][0])
    ranges.append(x[k][1])
    r_depths.append(x[k][2])
    water_depths.append(x[k][3])

fig = plt.figure(figsize=(10,3))

plt.subplot(1,4,1)
plt.plot(source_depths, marker='o')

plt.subplot(1,4,2)
plt.plot(ranges, marker='o')

plt.subplot(1,4,3)
plt.plot(r_depths, marker='o')

plt.subplot(1,4,4)
plt.plot(water_depths, marker='o')

plt.tight_layout()

## No water depth


`ray_ids = list(times.keys())[7:-6]`

In [ ]:
## Open Observed Arrival Times
Tx_idx = 0

with open(
    f"{os.environ['NESCAFE23_DATADIR']}dataproducts/peak_locations.pkl", "rb"
) as f:
    locations = pickle.load(f)

times = locations["times"]
angles = locations["angles"]
snrs = locations["snrs"]




In [ ]:
ray_sets = [
    slice(7, -14),
    slice(7, -10),
    slice(7, -6),
    slice(7, None)
]

ray_id_sets = []
for ray_set in ray_sets:
    ray_id_sets.append(list(times.keys())[ray_set])

In [ ]:
H

In [ ]:
source_depths = []
ranges = []
r_depths = []

Es = []
ys = []
dxs = []
Ps = []
xs = []

for ray_set_idx in range(4):
    
    ray_ids = ray_id_sets[ray_set_idx]
    difference_keys = list(combinations(ray_ids, 2))
    Delta_Ts_obs = np.ones(len(difference_keys)) * np.nan
    
    for k, diff_key in enumerate(difference_keys):
        Delta_Ts_obs[k] = times[diff_key[0]][Tx_idx] - times[diff_key[1]][Tx_idx]

    
    ## Create model environment object
    full_field = tools.compute_filtered_fctd_grid(z_smooth=300, r_smooth=5000)
    
    front_location = 35e3
    
    envi_field = full_field.assign_coords({
        "range": (full_field.range.values + front_location)
    })
    c = envi_field["c"]
    
    bathy = xr.DataArray(
        np.linspace(H, H, 100),
        dims=["range"],
        coords={"range": np.linspace(0, 200e3, 100)},
    )
    
    envi = pr.OceanEnvironment2D(sound_speed=c, bathymetry=bathy)

    ## Initial Guess and Priors

    source_depth0 = 1000  # [m]
    reciever_range0 = 153e3  # [m]
    reciever_depth0 = 1000  # [m]
    water_depth0 = 5000 # [m]
    
    S = np.diag([500, 4000, 500]) ** 2
    
    # 500 [m] for source depth
    # 1 [km] for range
    # 500 [m] for reciever range
    # 500 [m] for water depth
    
    R = 2 * np.diag([(20e-3)] * len(difference_keys)) ** 2
    # observation error of 20 [ms]

    x0 = np.array([source_depth0, reciever_range0, reciever_depth0])

    Delta_Ts_sim = []
    E = []
    y = []
    dx = []
    P = []
    rays = []
    eigs = []
    x = [x0]
    
    for k in range(5):
        print(
            f"Iteration {k}: Source Depth = {x[k][0]}, Range = {x[k][1]}, Reciever Depth = {x[k][2]}"
        )
        
        Delta_Ts_simi, Ei, yi, dxi, Pi, raysi, eigsi= source_localization_iteration(
            x[k], S, R, envi, ray_ids, Delta_Ts_obs
        )
    
        Delta_Ts_sim.append(Delta_Ts_simi)
        E.append(Ei)
        y.append(yi)
        dx.append(dxi)
        P.append(Pi)
        rays.append(raysi)
        eigs.append(eigsi)
        
        # compute update for next iteration
        x.append(x[k] + dx[k])

    source_depths.append([])
    ranges.append([])
    r_depths.append([])
    
    Es.append([])
    ys.append([])
    dxs.append([])
    Ps.append([])
    xs.append([])

    for k in range(5):
        source_depths[ray_set_idx].append(x[k][0])
        ranges[ray_set_idx].append(x[k][1])
        r_depths[ray_set_idx].append(x[k][2])

        Es[ray_set_idx].append(E[k])
        ys[ray_set_idx].append(y[k])
        dxs[ray_set_idx].append(dx[k])
        Ps[ray_set_idx].append(P[k])
        xs[ray_set_idx].append(x[k])

In [ ]:
P_diag = np.diagonal(np.array(Ps), axis1=2, axis2=3)
P_diag.shape

In [ ]:
P_diag = np.diagonal(np.array(Ps), axis1=2, axis2=3)

fig = plt.figure(figsize=(10,4))
for H_idx in range(1,4):
    plt.subplot(1,3,1)
    plt.errorbar(np.arange(len(source_depths[H_idx])), source_depths[H_idx], np.sqrt(P_diag[H_idx,:,0]), marker='o', markersize=4, capsize=3)
    plt.xlabel('iteration number')

    plt.ylim([2000,3000])
    plt.subplot(1,3,2)
    plt.errorbar(np.arange(len(ranges[H_idx])), ranges[H_idx], np.sqrt(P_diag[H_idx, :,1]), markersize=4, capsize=3)
    plt.xlabel('iteration number')


    plt.subplot(1,3,3)
    plt.errorbar(np.arange(len(r_depths[H_idx])), r_depths[H_idx], np.sqrt(P_diag[H_idx,:,2]), markersize=4, capsize=3, label=f'{len(ray_id_sets[H_idx])} Arrivals')
    
    plt.xlabel('iteration number')
    if H_idx==3:
        plt.legend()


plt.tight_layout()

In [ ]:
plt.plot(ys[0][-1])

In [ ]:
for k in range(4):
    print(f'{np.mean(ys[k][-1])} ± {np.std(ys[k][-1])}')

In [ ]:
#_ = plt.hist(ys[0][-1], alpha=0.2, bins=30)
_ = plt.hist(ys[1][-1], alpha=0.2, bins=30)
_ = plt.hist(ys[2][-1], alpha=0.2, bins=30)
#_ = plt.hist(ys[3][-1], alpha=0.2, bins=30)

In [ ]:
fig = plt.figure(figsize=(10,4))
for H_idx in range(1):
    plt.subplot(1,3,1)
    plt.plot(source_depths[H_idx], marker='o')
    
    plt.subplot(1,3,2)
    plt.plot(ranges[H_idx], marker='o')
    
    plt.subplot(1,3,3)
    plt.plot(r_depths[H_idx], marker='o', label=f'H={Hs[H_idx]} [m]')
    
    plt.tight_layout()

    plt.xlabel('iteration number')
    if H_idx==2:
        plt.legend()

In [ ]:
idx].append(x[k][0])
        ranges.append(x[k][1])
        r_depths.append(x[k][2])

In [ ]:
## Initial Guess and Priors

source_depth0 = 1000  # [m]
reciever_range0 = 153e3  # [m]
reciever_depth0 = 1000  # [m]
water_depth0 = 5000 # [m]

S = np.diag([500, 4000, 500]) ** 2

# 500 [m] for source depth
# 1 [km] for range
# 500 [m] for reciever range
# 500 [m] for water depth

R = 2 * np.diag([(20e-3)] * len(difference_keys)) ** 2
# observation error of 20 [ms]

In [ ]:
x0 = np.array([source_depth0, reciever_range0, reciever_depth0])

Delta_Ts_sim = []
E = []
y = []
dx = []
P = []
rays = []
eigs = []
x = [x0]

for k in range(5):
    print(
        f"Iteration {k}: Source Depth = {x[k][0]}, Range = {x[k][1]}, Reciever Depth = {x[k][2]}"
    )
    
    Delta_Ts_simi, Ei, yi, dxi, Pi, raysi, eigsi= source_localization_iteration(
        x[k], S, R, envi, ray_ids, Delta_Ts_obs
    )

    Delta_Ts_sim.append(Delta_Ts_simi)
    E.append(Ei)
    y.append(yi)
    dx.append(dxi)
    P.append(Pi)
    rays.append(raysi)
    eigs.append(eigsi)
    
    # compute update for next iteration
    x.append(x[k] + dx[k])

In [ ]:
source_depths = []
ranges = []
r_depths = []

for k in range(5):
    source_depths.append(x[k][0])
    ranges.append(x[k][1])
    r_depths.append(x[k][2])

fig = plt.figure(figsize=(10,3))

plt.subplot(1,3,1)
plt.plot(source_depths, marker='o')

plt.subplot(1,3,2)
plt.plot(ranges, marker='o')

plt.subplot(1,3,3)
plt.plot(r_depths, marker='o')

plt.tight_layout()